# WM-811K Wafer Map Defect Pattern Recognition
## Phase 1: Exploratory Data Analysis (EDA) & Domain Understanding

### 1. Project Background
In semiconductor and disk drive manufacturing (e.g., Seagate, Sony, TSMC), integrated circuits and magnetic heads are fabricated on circular silicon wafers. Each wafer contains hundreds to thousands of individual chips called **Dies**.

After wafer fabrication, automated electrical testing (Wafer Prober / AOI) tests every die:
- **0**: Empty background (outside the circular wafer boundary)
- **1**: Normal die (Passed electrical/optical test)
- **2**: Defective die (Failed test)

When defective dies (value 2) form spatial clusters or geometrical patterns (e.g., Scratch, Edge-Ring, Center), they indicate specific equipment or process malfunctions. Identifying these patterns in real-time saves millions of dollars in scrapped silicon.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline


### 2. Loading the Raw Dataset
The dataset is stored in `data/LSWMD.pkl` (approx. 2.09 GB). Let's load and inspect its shape and schema.


In [ ]:
print('Loading LSWMD.pkl...')
df = pd.read_pickle('../data/LSWMD.pkl')
print(f'Total Wafers: {len(df):,}')
df.info()
df.head(3)


### 3. Parsing Failure Labels & Handling Unlabeled Data
In the raw dataset:
- `failureType` is stored as nested numpy arrays (e.g. `array([['Center']], dtype='<U6')`).
- Wafers without defect labels have empty arrays `array([], shape=(0, 0))`.


In [ ]:
def parse_failure_label(val):
    if isinstance(val, np.ndarray) and val.size > 0:
        return val[0][0]
    return 'Unlabeled'

df['failureLabel'] = df['failureType'].apply(parse_failure_label)

label_counts = df['failureLabel'].value_counts()
print('Failure Label Breakdown:')
print(label_counts)


### 4. Class Distribution & Extreme Imbalance
Let's filter for labeled wafers (172,950 wafers) and visualize the severe imbalance between normal wafers (`none`) and actual defect classes.


In [ ]:
labeled_df = df[df['failureLabel'] != 'Unlabeled'].copy()

plt.figure(figsize=(10, 5))
counts = labeled_df['failureLabel'].value_counts()
bars = plt.bar(counts.index, counts.values, color='steelblue')
plt.title('WM-811K Labeled Class Distribution (Log Scale)', fontsize=14, fontweight='bold')
plt.xlabel('Defect Type')
plt.ylabel('Count (Log Scale)')
plt.yscale('log')
plt.xticks(rotation=45)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval * 1.1, f'{yval:,}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()


### 5. Multi-product Dimension Analysis (Varying Shapes)
Because wafers come from a multi-product foundry, the number of dies per wafer varies widely, resulting in 346 distinct matrix dimensions!


In [ ]:
labeled_df['waferDim'] = labeled_df['waferMap'].apply(lambda x: x.shape)
print(f'Total Unique Wafer Dimensions: {labeled_df["waferDim"].nunique()}')
print('\nTop 10 Most Common Dimensions:')
print(labeled_df['waferDim'].value_counts().head(10))


### 6. Visualizing the 9 Defect Classes
Let's plot representative examples of each defect class to understand their spatial patterns.


In [ ]:
classes = ['none', 'Center', 'Donut', 'Edge-Loc', 'Edge-Ring', 'Loc', 'Random', 'Scratch', 'Near-full']

fig, axes = plt.subplots(3, 3, figsize=(11, 11))
axes = axes.flatten()

for i, cls in enumerate(classes):
    sample = labeled_df[labeled_df['failureLabel'] == cls].iloc[0]
    img = sample['waferMap']
    axes[i].imshow(img, cmap='viridis')
    axes[i].set_title(f'Class: {cls}\nShape: {img.shape}', fontsize=12, fontweight='bold')
    axes[i].axis('off')

plt.tight_layout()
plt.show()
